# DUDE decision rule

We use SymPy to build the rule.

In [ ]:
from IPython.display import display

from estimator.matrices import get_slot, read_matrices

import numpy as np
from sympy import *
from sympy.abc import x, y, z
init_printing(use_latex=True)

matrices = read_matrices("matrices_by_quality.json")

z_i = 0
Q = 25
DEBUG = False

# loss function
loss_fn = Matrix(4, 4, lambda i, j: 0 if i == j else 1)

n_A, n_C, n_G, n_T = symbols('n_A n_C n_G n_T') 

# counters
mT = Matrix([[n_A, n_C, n_G, n_T]])

pi = Matrix(get_slot(matrices, 'substitutions', Q)['channel'])
pi_1 = Matrix(get_slot(matrices, 'substitutions', Q)['inverse'])

if DEBUG:
    display(loss_fn)
    display(mT)
    display(pi)
    display(pi_1)

# this is used to obtain the value from the sympy expression
polyKeys = [
    (1, 0, 0, 0),  # A
    (0, 1, 0, 0),  # C
    (0, 0, 1, 0),  # G
    (0, 0, 0, 1)   # T
]

values = []

for i in range(4):
    rule = (mT * pi_1) * (loss_fn.col(i).multiply_elementwise(pi.col(z_i)))
    display(rule)
    pol = rule[0].as_poly().as_dict()
    inner_values = []
    for j in range(4):
        inner_values.append(pol[polyKeys[j]])    
    values.append(inner_values)
    
###################################################################################

# min calc

# x = A and z = C
pA = symbols('pA')

a_A = abs(values[0][0])
a_C = abs(values[0][1])

c_A = abs(values[1][0])
c_C = abs(values[1][1])

res = ((c_A - c_C) / (a_C - a_A))

if DEBUG:
    print(c_A, c_C, a_C, a_A, res, sep="  |  ")

val_1 = solve((pA/(1-pA)) - res, pA)[0]
if DEBUG:
    display(val_1)

################################################

# x = A and z = G
pA = None
pA = symbols('pA')

a_A = abs(values[0][0])
a_G = abs(values[0][2])

g_A = abs(values[2][0])
g_G = abs(values[2][2])

res = ((g_A - g_G) / (a_G - a_A))

if DEBUG:
    print(g_A, g_G, a_G, a_A, res, sep="  |  ")

val_2 = solve((pA/(1-pA)) - res, pA)[0]

if DEBUG:
    display(val_2)

################################################

# x = A and z = T
pA = None
pA = symbols('pA')

a_A = abs(values[0][0])
a_T = abs(values[0][3])

t_A = abs(values[3][0])
t_T = abs(values[3][3])

res = ((t_A - t_T) / (a_T - a_A))
if DEBUG:
    print(t_A, t_T, a_T, a_A, res, sep="  |  ")

val_3 = solve((pA/(1-pA)) - res, pA)[0]
if DEBUG:
    display(val_3)

print('\nthe min value that pA needs to choose A is: ' + str(min([val_1, val_2, val_3])))